In [1]:
%pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import math
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
csv_files = list(Path(".").glob("*.csv"))

print("CSV files found:")
print()

for file in csv_files:
    print("-", file.name)

print()
print("Total CSV files:", len(csv_files))

CSV files found:

- lifecycle_events.csv
- organizations.csv
- product_units_seed.csv
- provenance_flags (3).csv
- users.csv
- verification_events.csv

Total CSV files: 6


In [5]:
organizations = pd.read_csv("organizations.csv")
verification_events = pd.read_csv("verification_events.csv")
lifecycle_events = pd.read_csv("lifecycle_events.csv")
product_units = pd.read_csv("product_units_seed.csv")
provenance_flags = pd.read_csv("provenance_flags (3).csv")
users = pd.read_csv("users.csv")

print("All CSV files loaded successfully!")

All CSV files loaded successfully!


In [6]:
print("========== ORGANIZATIONS ==========")
print(organizations.columns.tolist())

print("\n========== VERIFICATION EVENTS ==========")
print(verification_events.columns.tolist())

print("\n========== LIFECYCLE EVENTS ==========")
print(lifecycle_events.columns.tolist())

print("\n========== PRODUCT UNITS ==========")
print(product_units.columns.tolist())

print("\n========== PROVENANCE FLAGS ==========")
print(provenance_flags.columns.tolist())

print("\n========== USERS ==========")
print(users.columns.tolist())

========== ORGANIZATIONS ==========
['id', 'name', 'org_type', 'created_at']

========== VERIFICATION EVENTS ==========
['id', 'product_unit_id', 'result', 'ip_hash', 'user_agent_hash', 'created_at']

========== LIFECYCLE EVENTS ==========
['id', 'product_unit_id', 'actor_id', 'actor_organization_id', 'event_type', 'location_label', 'latitude', 'longitude', 'timestamp', 'event_hash', 'previous_event_hash', 'tx_hash', 'chain_status', 'idempotency_key', 'metadata', 'created_at']

========== PRODUCT UNITS ==========
['id', 'product_hash', 'qr_public_token', 'gtin', 'batch_no', 'serial_no', 'product_name', 'expiry_date', 'manufacturer_org_id', 'current_custodian_org_id', 'status', 'risk_score', 'risk_level', 'scan_count', 'last_verified_at', 'created_at']

========== PROVENANCE FLAGS ==========
['id', 'product_unit_id', 'event_id', 'risk_score', 'reason', 'feature_values', 'model_version', 'created_at']

========== USERS ==========
['id', 'email', 'password_hash', 'role', 'organization_id'

In [7]:
print("========== LIFECYCLE EVENTS ==========")
display(lifecycle_events.head(10))

print("\n========== PRODUCT UNITS ==========")
display(product_units.head(10))

print("\n========== VERIFICATION EVENTS ==========")
display(verification_events.head(10))

print("\n========== PROVENANCE FLAGS ==========")
display(provenance_flags.head(10))

========== LIFECYCLE EVENTS ==========


,id,product_unit_id,actor_id,actor_organization_id,event_type,location_label,latitude,longitude,timestamp,event_hash,previous_event_hash,tx_hash,chain_status,idempotency_key,metadata,created_at
0,1,1,24,5,manufactured,Chicago Regional Warehouse,41.8781,-87.6298,2024-01-13 00:00:00+00,98585201e0f91f050f1483d3c1ac0f8a90967279e11bcd...,0000000000000000000000000000000000000000000000...,0x336f85821f33795c929776f120c21263986c40a481b4...,confirmed,unit-1-manufactured-20240113T000000-3286,"{""batch_no"": ""B9935"", ""line"": ""A""}",2024-01-13 00:05:00+00
1,2,1,12,4,sold,New York Fulfillment Center,40.7128,-74.0060,2024-08-01 05:00:00+00,a5d0992fc32b4192857c7dbbe3c95a9a33fd1487d843f1...,0000000000000000000000000000000000000000000000...,0x6bbcaae97c9c33eec47f5cb5a24987ba1a4925cc00fe...,confirmed,unit-1-sold-20240801T050000-2402,{},2024-08-01 05:02:00+00
2,3,1,14,3,manufactured,Mumbai Inspection Center,19.0760,72.8777,2025-01-07 11:00:00+00,2c33fdf7ddbdd4564fccbf203a4139e54ba582c1887723...,a5d0992fc32b4192857c7dbbe3c95a9a33fd1487d843f1...,NaN,pending,unit-1-manufactured-20250107T110000-6995,{},2025-01-07 11:04:00+00
3,4,1,17,5,manufactured,Mexico City Assembly Plant,19.4326,-99.1332,2025-05-15 09:00:00+00,a10aa79c84f6cb3f30d5cd7eab43064c1888daa8bea013...,2c33fdf7ddbdd4564fccbf203a4139e54ba582c1887723...,0x3e94a86f1fa8f987b9e02f1c8bd874a8cf122841e6fd...,failed,unit-1-manufactured-20250515T090000-8054,{},2025-05-15 09:00:00+00
4,5,2,2,1,manufactured,Los Angeles Port Terminal,33.7406,-118.2706,2024-08-04 00:00:00+00,32d102697fcac0f6be704b32ce5c246b1994a7c2458ef0...,0000000000000000000000000000000000000000000000...,0x08c48958d5f8eb4c1c284d2a5b425818d88c3a94b2a4...,confirmed,unit-2-manufactured-20240804T000000-4811,"{""batch_no"": ""B1434"", ""line"": ""C""}",2024-08-04 00:04:00+00
5,6,2,7,9,inspected,Dallas Retail Store #12,32.7767,-96.7970,2024-08-05 08:00:00+00,e9b833a13f7be9a3f7fa6ca4551a1bec34ca55269e75dc...,32d102697fcac0f6be704b32ce5c246b1994a7c2458ef0...,0xae8a7318f4ce4da0c02a891e99df2adf0238553039d5...,confirmed,unit-2-inspected-20240805T080000-8359,"{""inspector_notes"": ""passed"", ""score"": 96.4}",2024-08-05 08:00:00+00
6,7,2,6,7,shipped,Hamburg Container Yard,53.5511,9.9937,2024-08-07 06:00:00+00,69a63a208b01de5cfb61993fc2d668e03afad8808cdbec...,e9b833a13f7be9a3f7fa6ca4551a1bec34ca55269e75dc...,0x628d719fb0151126ce185f8eb3d1eb38bf3b3a4a75ba...,confirmed,unit-2-shipped-20240807T060000-3547,"{""carrier"": ""DHL"", ""container_id"": ""CNT207175""}",2024-08-07 06:01:00+00
7,8,2,3,7,received,Los Angeles Port Terminal,33.7406,-118.2706,2024-08-10 03:00:00+00,0e77e30bf89275761f2044baeeaa9f2b56d2eb0c6d0c63...,69a63a208b01de5cfb61993fc2d668e03afad8808cdbec...,0xb396fac7ba5f2f0ac52cb533f87c1dbabd481380384b...,confirmed,unit-2-received-20240810T030000-6635,"{""dock"": ""D2"", ""condition"": ""good""}",2024-08-10 03:01:00+00
8,9,2,15,9,sold,Los Angeles Port Terminal,33.7406,-118.2706,2024-08-12 16:00:00+00,1df68a1816ed251eecaf620121823087c5e1f697c01406...,0e77e30bf89275761f2044baeeaa9f2b56d2eb0c6d0c63...,0x83192932ba1adf56dd23e289539cbf3c1daab2ed495c...,confirmed,unit-2-sold-20240812T160000-2291,"{""order_id"": ""ORD92397"", ""channel"": ""wholesale""}",2024-08-12 16:05:00+00
9,10,2,11,2,shipped,Dallas Retail Store #12,32.7767,-96.7970,2024-10-25 09:00:00+00,6e5a466b698305da3bcb540357d897a965640aa9fb0c2a...,5bf6ec75cb9e51c68d7966f51033f6f12b1a1fffc1c803...,0xdc028e15c07b8c175ffe1fed9141d66ffecbcf3fb1da...,confirmed,unit-2-shipped-20241025T090000-1436,{},2024-10-25 09:04:00+00



========== PRODUCT UNITS ==========


,id,product_hash,qr_public_token,gtin,batch_no,serial_no,product_name,expiry_date,manufacturer_org_id,current_custodian_org_id,status,risk_score,risk_level,scan_count,last_verified_at,created_at
0,1,c6a65e9d50f9c257392b9bedcfec8fda0fc745242de8b8...,TpigTHKbfoLISRABr1VjArnVgxwvqcCh,90268310679883,B8428-U56,SN-200604-000001-479,ORS Rehydration Sachet,2027-07-11,23,22,delivered,60.91,medium,2,2025-10-18 17:34:40,2026-05-21 18:49:40
1,2,dec9e2fac2373f2d3ea98c0d91aeff66068d7bf21ff50e...,obtqn62tOy4CqpIqK3yn9FfcgMXAdx9G,98076526144982,B9201-Y32,SRL-197106-000002-208,Insulin Regular Vial,2028-07-26,17,20,created,37.39,medium,5,2026-01-04 16:02:40,2025-05-24 04:13:40
2,3,7329b45772f55ab1f3ce0e923edade81322101d08f2bcb...,ooJeTY8HhO6kGL75UQSyPx3BpdbIKaRdebuFrEHS,85244063480660,B6559-Z23,SN-198508-000003-296,Omeprazole 20mg Capsules,2027-03-03,14,6,in_transit,24.98,low,0,NaN,2025-10-12 22:16:40
3,4,f6a930e19ad9a2420f4b83cc0b55ad2b430d5cdff6bbac...,50kEnydx9qWCA79ISjs8JHUdKF0j7elK,62182564559272,B7484-E95,SRL-198208-000004-407,Salbutamol Inhaler,2028-11-18,3,1,in_transit,99.85,high,1,2026-07-06 04:13:40,2026-08-07 08:46:40
4,5,af7fc78eef6d17fe85bf3eb5907338b3451b62850e05af...,ZRL9OaYsP6ihgIqLSmNqE40fAraVNqTIAae24HZK,11387882217846,B3532-H30,SN-201110-000005-522,Amoxicillin 500mg Capsules,2027-09-16,14,22,sold,81.07,high,12,2026-05-21 09:17:40,2025-05-29 01:47:40
5,6,f82369ba05157aeddd78aa787db82e63c6812ff4e31c62...,nOyreVvFQ0ub2qJ8cKvWB9h3lcBGYQ6T,14996619563651,B7211-V32,SRL-200805-000006-682,Amlodipine 5mg Tablets,2029-02-10,1,10,in_transit,42.99,medium,3,2026-05-24 23:39:40,2025-07-31 16:59:40
6,7,9da0a21fc0a5b845e56bfb3203adca147e838d96b34aeb...,FNuYUPnmbpD0ezNmREpOaUVgAk7Gdp0CXP9K63LS,58587422675620,B5681-H44,PU-202007-000007-427,Multivitamin Syrup,2026-06-25,8,13,delivered,NaN,NaN,4,2026-08-11 09:16:40,2026-02-08 03:12:40
7,8,7b6539920d6b994d3c6ea88871f6dca3dbbef7884ce63c...,b2JD6sy3ZHTX3EqEyPXS058H4KPfA1lquCu2r6AZDUd7ne7c,2862419423697,B3683-J23,SRL-197412-000008-126,Amlodipine 5mg Tablets,2027-10-30,13,23,created,59.21,medium,7,2025-11-08 12:19:40,2025-11-26 14:47:40
8,9,823a7a7f04b45f783e4bea5509e7756f1eef14e8b159ab...,wFv0Yg7NZRBT7qYHDBTq0Zf2pCLxb0lnXv2Rra6fSEVQOEXf,26354547372211,B2958-X78,SRL-199312-000009-289,Omeprazole 20mg Capsules,2028-02-18,9,24,delivered,76.02,high,1,2026-03-11 10:50:40,2026-09-03 17:35:40
9,10,3495a2dd1fbef24ee01af07a965cdbdf8ebd5404e0e236...,8IRh1E2JDBld6DYyeNdjIs9hVLXnGBB1,93667922611004,B8699-J14,SN-197109-000010-395,Pantoprazole 40mg Tablets,2028-01-18,3,22,in_transit,26.46,low,5,2025-12-07 08:28:40,2026-06-27 07:17:40



========== VERIFICATION EVENTS ==========


,id,product_unit_id,result,ip_hash,user_agent_hash,created_at
0,1,82,success,f5b73c244f2c7f90da119cd5d5ebca35550f125a27936e...,ff431625483a89f24940504ef3aea91fee5af098ee9214...,2025-01-01T08:27:00+00:00
1,2,87,failed,68fca94fd75d6cd1e7b49cbaa9689cb2906150c3f1e2fc...,726c1edb2893744a0ae3173c01e21b06e4f145ca269228...,2025-01-01T08:51:00+00:00
2,3,28,success,9ff9338fa6acff0a518b97d951dc763f1eb0c89d0b8fd2...,11c229eb81cf7e4fb439898266da372925a2bfbd63c227...,2025-01-01T11:51:00+00:00
3,4,70,success,0dddf9437bfdb1f3bb521d170e95c6da5a42220acf3710...,6ace6919479fa43137c1a477ae7568e02614d4c0d69671...,2025-01-01T15:06:00+00:00
4,5,104,success,e575bb842e4df2444b00a59dc992aaad8ce71d70803b30...,65d89b50e7c05807cce2edea82f21b25d10b4002af550e...,2025-01-01T18:22:00+00:00
5,6,44,success,b6bf6db63c0ff02d9204ea3f03669e6dd24920bcd85f59...,81193016c125855ba170923e97d63d387f905f974e9498...,2025-01-01T19:30:00+00:00
6,7,104,success,aaa7b2fd4f1863e61d8b2cde11fc6fa8f8bf05344d2f3b...,f178a4e862e8c252377e61d15a91206987a293fa4d32ee...,2025-01-01T19:51:00+00:00
7,8,71,success,ae69b94918e1349e8b9b9961d1f6a1e0d44664cea208c5...,fb22a0b9bfd964d3a588e89c82c2cf7c0f6676e4be56f2...,2025-01-01T20:41:00+00:00
8,9,91,success,59335a3e28ee5c2b7cbaf32106c87ab62d0553ab05e1a0...,c2f65e01305a3dff7fbccf287b3dceadcaecb02543fa7a...,2025-01-02T00:20:00+00:00
9,10,30,invalid,2122e6d5179c6e0299b691f3cca205868203919676af8b...,fb92faf99ceb53083259adff9aa0dfabbe5bd7e5bf873c...,2025-01-02T01:02:00+00:00



========== PROVENANCE FLAGS ==========


,id,product_unit_id,event_id,risk_score,reason,feature_values,model_version,created_at
0,1,166,203.0,7.24,Weight variance outside tolerance for declared...,"{""temp_excursion_c"":454.85,""scan_velocity_kmh""...",prov-risk-v1.0.0,2025-11-04 15:28:58+00
1,2,215,NaN,42.45,Origin country mismatch with declared manufact...,"{""checksum_valid"":false,""scan_velocity_kmh"":24...",prov-risk-v1.1.2,2024-03-13 08:43:41+00
2,3,286,149.0,54.07,Origin country mismatch with declared manufact...,"{""supplier_risk_tier"":""high"",""time_since_last_...",prov-risk-v2.1.0,2024-04-07 12:56:33+00
3,4,289,NaN,68.04,Cross-border movement without customs event,"{""label_print_delta_days"":-15,""packaging_hash_...",prov-risk-v2.0.1,2025-04-11 10:05:54+00
4,5,269,176.0,28.79,Duplicate unit ID detected across regions,"{""supplier_risk_tier"":""low"",""reseller_risk_sco...",prov-risk-v1.1.2,2026-01-29 13:51:56+00
5,6,216,NaN,76.46,Origin country mismatch with declared manufact...,"{""temp_excursion_c"":398.45,""packaging_hash_sim...",prov-risk-v2.0.0-beta,2024-04-10 23:22:29+00
6,7,32,159.0,99.31,Anomalous reseller account activity,"{""reseller_risk_score"":0.355,""temp_excursion_c...",NaN,2025-03-22 12:14:20+00
7,8,67,204.0,87.14,Duplicate unit ID detected across regions,"{""label_print_delta_days"":26,""reseller_risk_sc...",NaN,2026-05-04 17:40:18+00
8,9,143,184.0,38.04,Temperature excursion detected during transit,"{""gps_deviation_km"":115.98,""time_since_last_ev...",NaN,2026-07-04 04:06:58+00
9,10,94,3.0,53.46,Model version confidence below acceptance thre...,"{""time_since_last_event_hrs"":647.7,""supplier_r...",prov-risk-v2.0.0-beta,2025-09-11 19:00:32+00


In [8]:
print("========== MISSING VALUES ==========")

print("\nLifecycle events:")
display(lifecycle_events.isnull().sum())

print("\nProduct units:")
display(product_units.isnull().sum())

print("\nVerification events:")
display(verification_events.isnull().sum())


print("\n========== DUPLICATE ROWS ==========")

print("Lifecycle duplicates:", lifecycle_events.duplicated().sum())
print("Product unit duplicates:", product_units.duplicated().sum())
print("Verification duplicates:", verification_events.duplicated().sum())

========== MISSING VALUES ==========

Lifecycle events:


id                        0
product_unit_id           0
actor_id                  0
actor_organization_id     0
event_type                0
location_label           20
latitude                 20
longitude                20
timestamp                 0
event_hash                0
previous_event_hash       0
tx_hash                  81
chain_status              0
idempotency_key           0
metadata                  0
created_at                0
dtype: int64


Product units:


id                           0
product_hash                 0
qr_public_token              0
gtin                         0
batch_no                     0
serial_no                    0
product_name                 0
expiry_date                 22
manufacturer_org_id          0
current_custodian_org_id     0
status                       0
risk_score                  16
risk_level                  16
scan_count                   0
last_verified_at            88
created_at                   0
dtype: int64


Verification events:


id                 0
product_unit_id    0
result             0
ip_hash            9
user_agent_hash    9
created_at         0
dtype: int64


========== DUPLICATE ROWS ==========
Lifecycle duplicates: 0
Product unit duplicates: 0
Verification duplicates: 0


In [9]:
lifecycle_events["timestamp"] = pd.to_datetime(
    lifecycle_events["timestamp"],
    errors="coerce",
    utc=True
)

verification_events["created_at"] = pd.to_datetime(
    verification_events["created_at"],
    errors="coerce",
    utc=True
)

if "created_at" in product_units.columns:
    product_units["created_at"] = pd.to_datetime(
        product_units["created_at"],
        errors="coerce",
        utc=True
    )

lifecycle_events = lifecycle_events.sort_values(
    ["product_unit_id", "timestamp"]
).reset_index(drop=True)

verification_events = verification_events.sort_values(
    ["product_unit_id", "created_at"]
).reset_index(drop=True)

print("Timestamp conversion and sorting complete!")

Timestamp conversion and sorting complete!


In [10]:
# ============================================================
# SUPPLYCHAINX - PROVENANCE ANOMALY FEATURES
# ============================================================

FEATURE_NAMES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

ORDER = {
    "manufactured": 0,
    "inspected": 1,
    "shipped": 2,
    "received": 3,
    "sold": 4
}

MAX_PLAUSIBLE_KMH = 900.0

TEMP_MIN = 18.0
TEMP_MAX = 26.0

SHOCK_THRESHOLD_G = 3.0

MAX_SENSOR_GAP_HOURS = 2.0

# Generic time-gap threshold from the project specification.
# The shipped -> received transition is given a 3-day expected
# maximum because the specification's worked example uses that
# transition-specific expectation.
DEFAULT_MAX_GAP_DAYS = 14.0

TIME_GAP_THRESHOLDS_DAYS = {
    ("shipped", "received"): 3.0
}


def safe_timestamp(value):
    """
    Convert a value to a timezone-aware pandas timestamp.
    """
    try:
        ts = pd.to_datetime(value, errors="coerce", utc=True)
        return ts
    except Exception:
        return pd.NaT


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two latitude/longitude points.
    """
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)

        radius = 6371.0

        phi1 = math.radians(lat1)
        phi2 = math.radians(lat2)

        dphi = math.radians(lat2 - lat1)
        dlambda = math.radians(lon2 - lon1)

        a = (
            math.sin(dphi / 2) ** 2
            + math.cos(phi1)
            * math.cos(phi2)
            * math.sin(dlambda / 2) ** 2
        )

        c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

        return radius * c

    except Exception:
        return 0.0


# ------------------------------------------------------------
# FEATURE 1 - SEQUENCE VIOLATION
# ------------------------------------------------------------

def feature_sequence_violation(event_data, prior_events):
    current_type = str(event_data.get("event_type", "")).lower()

    if current_type not in ORDER:
        return False

    current_index = ORDER[current_type]

    for event in prior_events:
        previous_type = str(event.get("event_type", "")).lower()

        if previous_type in ORDER:
            if ORDER[previous_type] > current_index:
                return True

    return False


# ------------------------------------------------------------
# FEATURE 2 - GEO IMPLAUSIBILITY
# ------------------------------------------------------------

def feature_geo_implausibility(event_data, prior_events):
    if not prior_events:
        return False

    previous = prior_events[-1]

    try:
        lat1 = float(previous.get("latitude"))
        lon1 = float(previous.get("longitude"))
        lat2 = float(event_data.get("latitude"))
        lon2 = float(event_data.get("longitude"))

        t1 = safe_timestamp(previous.get("timestamp"))
        t2 = safe_timestamp(event_data.get("timestamp"))

        if pd.isna(t1) or pd.isna(t2):
            return False

        elapsed_hours = (t2 - t1).total_seconds() / 3600.0

        if elapsed_hours <= 0:
            return False

        distance = haversine_km(lat1, lon1, lat2, lon2)

        speed = distance / elapsed_hours

        return speed > MAX_PLAUSIBLE_KMH

    except Exception:
        return False


# ------------------------------------------------------------
# FEATURE 3 - DUPLICATE SCAN COUNT
# ------------------------------------------------------------

def feature_duplicate_scan_count(event_data, prior_events=None):
    count = event_data.get("_duplicate_scan_count", 0)

    try:
        return int(count)
    except Exception:
        return 0


# ------------------------------------------------------------
# FEATURE 4 - TIME GAP ANOMALY
# ------------------------------------------------------------

def feature_time_gap_anomaly(event_data, prior_events):
    if not prior_events:
        return False

    previous = prior_events[-1]

    try:
        previous_time = safe_timestamp(previous.get("timestamp"))
        current_time = safe_timestamp(event_data.get("timestamp"))

        if pd.isna(previous_time) or pd.isna(current_time):
            return False

        gap_seconds = (current_time - previous_time).total_seconds()

        if gap_seconds < 0:
            return True

        if gap_seconds < 60:
            return True

        previous_type = str(
            previous.get("event_type", "")
        ).lower()

        current_type = str(
            event_data.get("event_type", "")
        ).lower()

        threshold_days = TIME_GAP_THRESHOLDS_DAYS.get(
            (previous_type, current_type),
            DEFAULT_MAX_GAP_DAYS
        )

        return gap_seconds > threshold_days * 24 * 3600

    except Exception:
        return False


# ------------------------------------------------------------
# FEATURE 5 - UNEXPECTED CUSTODIAN
# ------------------------------------------------------------

def feature_unexpected_custodian(event_data):
    actor = event_data.get("actor_organization_id")
    custodian = event_data.get("_current_custodian_org_id")

    if actor is None or custodian is None:
        return False

    return str(actor) != str(custodian)


# ------------------------------------------------------------
# FEATURE 6 - TEMPERATURE VIOLATION
# ------------------------------------------------------------

def feature_temperature_violation(iot_readings):
    for reading in iot_readings:
        try:
            temperature = float(reading.get("temperature"))

            if temperature < TEMP_MIN or temperature > TEMP_MAX:
                return True

        except Exception:
            continue

    return False


# ------------------------------------------------------------
# FEATURE 7 - TEMPERATURE DURATION
# ------------------------------------------------------------

def feature_temperature_duration(iot_readings):
    if len(iot_readings) < 2:
        return 0.0

    readings = []

    for reading in iot_readings:
        try:
            timestamp = safe_timestamp(reading.get("timestamp"))
            temperature = float(reading.get("temperature"))

            if pd.isna(timestamp):
                continue

            readings.append(
                (timestamp, temperature)
            )

        except Exception:
            continue

    readings.sort(key=lambda x: x[0])

    duration_hours = 0.0

    for i in range(1, len(readings)):
        previous_time, previous_temp = readings[i - 1]
        current_time, current_temp = readings[i]

        previous_bad = (
            previous_temp < TEMP_MIN
            or previous_temp > TEMP_MAX
        )

        current_bad = (
            current_temp < TEMP_MIN
            or current_temp > TEMP_MAX
        )

        if previous_bad and current_bad:

            gap_hours = (
                current_time - previous_time
            ).total_seconds() / 3600.0

            if gap_hours > 0:
                duration_hours += gap_hours

    return min(5.0, duration_hours)


# ------------------------------------------------------------
# FEATURE 8 - SHOCK ANOMALY
# ------------------------------------------------------------

def feature_shock_anomaly(iot_readings):
    for reading in iot_readings:
        try:
            acceleration = float(
                reading.get("accel_magnitude")
            )

            if acceleration > SHOCK_THRESHOLD_G:
                return True

        except Exception:
            continue

    return False


# ------------------------------------------------------------
# FEATURE 9 - SENSOR GAP
# ------------------------------------------------------------

def feature_sensor_gap(event_data, iot_readings):
    if not iot_readings:
        return False

    device_id = event_data.get("_iot_device_id")
    status = str(
        event_data.get("_unit_status", "")
    ).lower()

    if device_id in [None, "", "nan", "None"]:
        return False

    if status != "in_transit":
        return False

    valid_times = []

    for reading in iot_readings:
        timestamp = safe_timestamp(
            reading.get("timestamp")
        )

        if not pd.isna(timestamp):
            valid_times.append(timestamp)

    if not valid_times:
        return False

    latest_reading = max(valid_times)

    event_time = safe_timestamp(
        event_data.get("timestamp")
    )

    if pd.isna(event_time):
        return False

    gap_hours = (
        event_time - latest_reading
    ).total_seconds() / 3600.0

    return gap_hours > MAX_SENSOR_GAP_HOURS


# ------------------------------------------------------------
# FEATURE 10 - SENSOR/PROVENANCE MISMATCH
# ------------------------------------------------------------

def feature_sensor_provenance_mismatch(
    event_data,
    iot_readings
):
    status = str(
        event_data.get("_unit_status", "")
    ).lower()

    if status not in ["delivered", "sold"]:
        return False

    accelerations = []

    for reading in iot_readings:
        try:
            value = float(
                reading.get("accel_magnitude")
            )

            accelerations.append(value)

        except Exception:
            continue

    if len(accelerations) < 3:
        return False

    average_motion = np.mean(accelerations)

    return average_motion > 1.5

In [13]:
print("Feature functions created:")
for feature in FEATURE_NAMES:
    print("✓", feature)

Feature functions created:
✓ sequence_violation
✓ geo_implausibility
✓ duplicate_scan_count
✓ time_gap_anomaly
✓ unexpected_custodian
✓ temperature_violation
✓ temperature_duration
✓ shock_anomaly
✓ sensor_gap
✓ sensor_provenance_mismatch


In [14]:
# ============================================================
# DETERMINISTIC SCORING ENGINE
# ============================================================

def calculate_contributions(feature_values):
    contributions = {}

    if feature_values["sequence_violation"]:
        contributions["sequence_violation"] = 45

    if feature_values["geo_implausibility"]:
        contributions["geo_implausibility"] = 45

    duplicate_count = feature_values["duplicate_scan_count"]

    if duplicate_count > 1:
        contributions["duplicate_scan_count"] = min(
            60,
            35 + (duplicate_count - 1) * 8
        )

    if feature_values["time_gap_anomaly"]:
        contributions["time_gap_anomaly"] = 25

    if feature_values["unexpected_custodian"]:
        contributions["unexpected_custodian"] = 40

    if feature_values["temperature_violation"]:
        contributions["temperature_violation"] = 40

    temperature_duration = feature_values[
        "temperature_duration"
    ]

    if temperature_duration > 0:
        contributions["temperature_duration"] = min(
            20,
            temperature_duration * 4
        )

    if feature_values["shock_anomaly"]:
        contributions["shock_anomaly"] = 35

    if feature_values["sensor_gap"]:
        contributions["sensor_gap"] = 20

    if feature_values["sensor_provenance_mismatch"]:
        contributions["sensor_provenance_mismatch"] = 45

    # IoT contribution cap
    iot_features = [
        "temperature_violation",
        "temperature_duration",
        "shock_anomaly",
        "sensor_gap",
        "sensor_provenance_mismatch"
    ]

    iot_total = sum(
        value
        for key, value in contributions.items()
        if key in iot_features
    )

    if iot_total > 60:
        scale = 60 / iot_total

        for key in list(contributions.keys()):
            if key in iot_features:
                contributions[key] = (
                    contributions[key] * scale
                )

    return contributions


def deterministic_score(feature_values):

    contributions = calculate_contributions(
        feature_values
    )

    score = min(
        100,
        round(sum(contributions.values()))
    )

    return score, contributions


def risk_level(score):

    if score >= 65:
        return "High"

    if score >= 35:
        return "Medium"

    return "Low"


def top_contributors(contributions):

    if not contributions:
        return []

    ordered = sorted(
        contributions.items(),
        key=lambda item: item[1],
        reverse=True
    )

    top = [ordered[0]]

    if len(ordered) > 1:

        first_value = ordered[0][1]
        second_value = ordered[1][1]

        if (
            first_value > 0
            and second_value > 0
            and abs(first_value - second_value) <= 5
        ):
            top.append(ordered[1])

    return top

In [15]:
# ============================================================
# REASON TEMPLATES
# ============================================================

def reason_sequence(event_data, prior_events):
    current_type = event_data.get(
        "event_type",
        "unknown"
    )

    later_stage = "unknown"

    current_index = ORDER.get(
        str(current_type).lower(),
        -1
    )

    for event in prior_events:
        event_type = str(
            event.get("event_type", "")
        ).lower()

        if event_type in ORDER:
            if ORDER[event_type] > current_index:
                later_stage = event_type
                break

    return (
        f"Event '{current_type}' logged out of order — "
        f"a later lifecycle stage ('{later_stage}') "
        f"was already recorded for this unit"
    )


def reason_geo(event_data, prior_events):
    if not prior_events:
        return "Implausible travel detected"

    previous = prior_events[-1]

    try:
        distance = haversine_km(
            previous["latitude"],
            previous["longitude"],
            event_data["latitude"],
            event_data["longitude"]
        )

        t1 = safe_timestamp(
            previous["timestamp"]
        )

        t2 = safe_timestamp(
            event_data["timestamp"]
        )

        hours = (
            t2 - t1
        ).total_seconds() / 3600

        speed = distance / hours

        return (
            f"Implausible travel: {distance:.0f} km "
            f"in {hours:.2f} hours "
            f"({speed:.0f} km/h, exceeds plausible maximum)"
        )

    except Exception:
        return "Implausible geographic movement detected"


def reason_duplicate(event_data):
    count = event_data.get(
        "_duplicate_scan_count",
        0
    )

    return (
        f"QR code scanned {count} times — "
        f"repeated scans may indicate label cloning or misuse"
    )


def reason_time_gap(event_data, prior_events):
    if not prior_events:
        return "Unusual lifecycle event time gap detected"

    previous = prior_events[-1]

    previous_type = previous.get(
        "event_type",
        "previous"
    )

    current_type = event_data.get(
        "event_type",
        "current"
    )

    t1 = safe_timestamp(
        previous.get("timestamp")
    )

    t2 = safe_timestamp(
        event_data.get("timestamp")
    )

    gap_days = (
        t2 - t1
    ).total_seconds() / 86400

    threshold = TIME_GAP_THRESHOLDS_DAYS.get(
        (
            str(previous_type).lower(),
            str(current_type).lower()
        ),
        DEFAULT_MAX_GAP_DAYS
    )

    return (
        f"Unusual gap between {previous_type} "
        f"and {current_type} events "
        f"({gap_days:.1f} days, expected < {threshold} days)"
    )


def reason_custodian(event_data):
    actor = event_data.get(
        "actor_organization_id"
    )

    custodian = event_data.get(
        "_current_custodian_org_id"
    )

    return (
        f"Event logged by organization {actor}, "
        f"but current custodian is organization {custodian}"
    )


def reason_temperature(iot_readings):
    for reading in iot_readings:

        try:
            temperature = float(
                reading["temperature"]
            )

            if (
                temperature < TEMP_MIN
                or temperature > TEMP_MAX
            ):
                return (
                    f"Temperature excursion: "
                    f"{temperature}°C outside configured "
                    f"range [{TEMP_MIN}°C, {TEMP_MAX}°C]"
                )

        except Exception:
            continue

    return "Temperature excursion detected"


def reason_temperature_duration(feature_value):

    return (
        f"Temperature remained outside permitted "
        f"range for approximately "
        f"{feature_value:.2f} hours"
    )


def reason_shock(iot_readings):

    for reading in iot_readings:

        try:
            acceleration = float(
                reading["accel_magnitude"]
            )

            if acceleration > SHOCK_THRESHOLD_G:

                return (
                    f"Shock/handling anomaly: "
                    f"acceleration magnitude "
                    f"{acceleration:.2f}g exceeds configured "
                    f"threshold {SHOCK_THRESHOLD_G}g"
                )

        except Exception:
            continue

    return "Shock/handling anomaly detected"


def reason_sensor_gap(event_data, iot_readings):

    if not iot_readings:
        return "No sensor readings received for an extended interval"

    latest = max(
        [
            safe_timestamp(x.get("timestamp"))
            for x in iot_readings
            if not pd.isna(
                safe_timestamp(x.get("timestamp"))
            )
        ]
    )

    event_time = safe_timestamp(
        event_data.get("timestamp")
    )

    gap_hours = (
        event_time - latest
    ).total_seconds() / 3600

    return (
        f"No sensor readings received for "
        f"{gap_hours:.1f} hours — possible sensor "
        f"failure or connectivity loss"
    )


def reason_sensor_mismatch(event_data):

    status = event_data.get(
        "_unit_status",
        "unknown"
    )

    return (
        f"Sensor evidence (sustained motion) "
        f"conflicts with recorded status '{status}'"
    )

In [16]:
# ============================================================
# FINAL AI INFERENCE FUNCTION
# ============================================================

def score_event(
    event_data: dict,
    prior_events: list,
    iot_readings: list
) -> dict:

    try:

        # ----------------------------------------------------
        # Calculate duplicate scans
        # ----------------------------------------------------

        if "_duplicate_scan_count" not in event_data:

            event_time = safe_timestamp(
                event_data.get("timestamp")
            )

            unit_id = event_data.get(
                "product_unit_id"
            )

            if unit_id is not None:

                unit_verifications = (
                    verification_events[
                        verification_events[
                            "product_unit_id"
                        ].astype(str)
                        == str(unit_id)
                    ]
                    .copy()
                )

                if "created_at" in unit_verifications.columns:
                    unit_verifications = (
                        unit_verifications[
                            unit_verifications[
                                "created_at"
                            ] <= event_time
                        ]
                    )

                event_data = dict(event_data)

                event_data[
                    "_duplicate_scan_count"
                ] = len(unit_verifications)

        # ----------------------------------------------------
        # Feature calculations
        # ----------------------------------------------------

        feature_values = {

            "sequence_violation":
                feature_sequence_violation(
                    event_data,
                    prior_events
                ),

            "geo_implausibility":
                feature_geo_implausibility(
                    event_data,
                    prior_events
                ),

            "duplicate_scan_count":
                feature_duplicate_scan_count(
                    event_data,
                    prior_events
                ),

            "time_gap_anomaly":
                feature_time_gap_anomaly(
                    event_data,
                    prior_events
                ),

            "unexpected_custodian":
                feature_unexpected_custodian(
                    event_data
                ),

            "temperature_violation":
                feature_temperature_violation(
                    iot_readings
                ),

            "temperature_duration":
                feature_temperature_duration(
                    iot_readings
                ),

            "shock_anomaly":
                feature_shock_anomaly(
                    iot_readings
                ),

            "sensor_gap":
                feature_sensor_gap(
                    event_data,
                    iot_readings
                ),

            "sensor_provenance_mismatch":
                feature_sensor_provenance_mismatch(
                    event_data,
                    iot_readings
                )
        }

        # ----------------------------------------------------
        # Score
        # ----------------------------------------------------

        score, contributions = deterministic_score(
            feature_values
        )

        level = risk_level(score)

        # ----------------------------------------------------
        # Reason
        # ----------------------------------------------------

        contributors = top_contributors(
            contributions
        )

        reasons = []

        for feature_name, contribution in contributors:

            if feature_name == "sequence_violation":

                reasons.append(
                    reason_sequence(
                        event_data,
                        prior_events
                    )
                )

            elif feature_name == "geo_implausibility":

                reasons.append(
                    reason_geo(
                        event_data,
                        prior_events
                    )
                )

            elif feature_name == "duplicate_scan_count":

                reasons.append(
                    reason_duplicate(
                        event_data
                    )
                )

            elif feature_name == "time_gap_anomaly":

                reasons.append(
                    reason_time_gap(
                        event_data,
                        prior_events
                    )
                )

            elif feature_name == "unexpected_custodian":

                reasons.append(
                    reason_custodian(
                        event_data
                    )
                )

            elif feature_name == "temperature_violation":

                reasons.append(
                    reason_temperature(
                        iot_readings
                    )
                )

            elif feature_name == "temperature_duration":

                reasons.append(
                    reason_temperature_duration(
                        feature_values[
                            "temperature_duration"
                        ]
                    )
                )

            elif feature_name == "shock_anomaly":

                reasons.append(
                    reason_shock(
                        iot_readings
                    )
                )

            elif feature_name == "sensor_gap":

                reasons.append(
                    reason_sensor_gap(
                        event_data,
                        iot_readings
                    )
                )

            elif feature_name == "sensor_provenance_mismatch":

                reasons.append(
                    reason_sensor_mismatch(
                        event_data
                    )
                )

        if reasons:

            reason = "; ".join(reasons)

        else:

            reason = (
                "No significant provenance anomaly detected"
            )

        return {
            "risk_score": int(score),
            "risk_level": level,
            "reason": reason,
            "feature_values": feature_values,
            "model_version": "deterministic_v1"
        }

    except Exception:

        # ----------------------------------------------------
        # Safe fallback
        # ----------------------------------------------------

        return {
            "risk_score": 0,
            "risk_level": "Low",
            "reason": "Scoring unavailable; safe fallback returned",
            "feature_values": {
                "sequence_violation": False,
                "geo_implausibility": False,
                "duplicate_scan_count": 0,
                "time_gap_anomaly": False,
                "unexpected_custodian": False,
                "temperature_violation": False,
                "temperature_duration": 0,
                "shock_anomaly": False,
                "sensor_gap": False,
                "sensor_provenance_mismatch": False
            },
            "model_version": "deterministic_v1_fallback"
        }

In [17]:
clean_event = {
    "product_unit_id": 1,
    "event_type": "manufactured",
    "actor_organization_id": 1,
    "_current_custodian_org_id": 1,
    "_unit_status": "manufactured",
    "latitude": 19.0760,
    "longitude": 72.8777,
    "timestamp": "2026-09-01T10:00:00Z",
    "_duplicate_scan_count": 1
}

result = score_event(
    clean_event,
    [],
    []
)

print(json.dumps(result, indent=4))

{
    "risk_score": 0,
    "risk_level": "Low",
    "reason": "No significant provenance anomaly detected",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 1,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0.0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [18]:
sequence_event = {
    "product_unit_id": 1,
    "event_type": "inspected",
    "actor_organization_id": 1,
    "_current_custodian_org_id": 1,
    "_unit_status": "inspected",
    "latitude": 19.0760,
    "longitude": 72.8777,
    "timestamp": "2026-09-02T10:00:00Z",
    "_duplicate_scan_count": 1
}

prior_events_test = [
    {
        "event_type": "manufactured",
        "timestamp": "2026-09-01T10:00:00Z",
        "latitude": 19.0760,
        "longitude": 72.8777
    },
    {
        "event_type": "shipped",
        "timestamp": "2026-09-01T12:00:00Z",
        "latitude": 19.0760,
        "longitude": 72.8777
    }
]

result = score_event(
    sequence_event,
    prior_events_test,
    []
)

print(json.dumps(result, indent=4))

{
    "risk_score": 45,
    "risk_level": "Medium",
    "reason": "Event 'inspected' logged out of order \u2014 a later lifecycle stage ('shipped') was already recorded for this unit",
    "feature_values": {
        "sequence_violation": true,
        "geo_implausibility": false,
        "duplicate_scan_count": 1,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0.0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [19]:
geo_event = {
    "product_unit_id": 1,
    "event_type": "received",
    "actor_organization_id": 2,
    "_current_custodian_org_id": 2,
    "_unit_status": "received",
    "latitude": 28.7041,
    "longitude": 77.1025,
    "timestamp": "2026-09-02T10:38:00Z",
    "_duplicate_scan_count": 1
}

prior_geo = [
    {
        "event_type": "shipped",
        "timestamp": "2026-09-02T10:00:00Z",
        "latitude": 19.0760,
        "longitude": 72.8777
    }
]

result = score_event(
    geo_event,
    prior_geo,
    []
)

print(json.dumps(result, indent=4))

{
    "risk_score": 45,
    "risk_level": "Medium",
    "reason": "Implausible travel: 1153 km in 0.63 hours (1821 km/h, exceeds plausible maximum)",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": true,
        "duplicate_scan_count": 1,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0.0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [20]:
duplicate_event = {
    "product_unit_id": 1,
    "event_type": "received",
    "actor_organization_id": 2,
    "_current_custodian_org_id": 2,
    "_unit_status": "received",
    "latitude": 28.7041,
    "longitude": 77.1025,
    "timestamp": "2026-09-03T10:00:00Z",
    "_duplicate_scan_count": 4
}

result = score_event(
    duplicate_event,
    [],
    []
)

print(json.dumps(result, indent=4))

{
    "risk_score": 59,
    "risk_level": "Medium",
    "reason": "QR code scanned 4 times \u2014 repeated scans may indicate label cloning or misuse",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 4,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0.0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [21]:
iot_test = [
    {
        "product_unit_id": 1,
        "timestamp": "2026-09-03T08:00:00Z",
        "temperature": 30.0,
        "accel_magnitude": 1.0
    },
    {
        "product_unit_id": 1,
        "timestamp": "2026-09-03T09:00:00Z",
        "temperature": 31.0,
        "accel_magnitude": 4.2
    }
]

iot_event = {
    "product_unit_id": 1,
    "event_type": "shipped",
    "actor_organization_id": 2,
    "_current_custodian_org_id": 2,
    "_unit_status": "in_transit",
    "latitude": 19.0760,
    "longitude": 72.8777,
    "timestamp": "2026-09-03T10:00:00Z",
    "_duplicate_scan_count": 1
}

result = score_event(
    iot_event,
    [],
    iot_test
)

print(json.dumps(result, indent=4))

{
    "risk_score": 60,
    "risk_level": "Medium",
    "reason": "Temperature excursion: 30.0\u00b0C outside configured range [18.0\u00b0C, 26.0\u00b0C]; Shock/handling anomaly: acceleration magnitude 4.20g exceeds configured threshold 3.0g",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 1,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": true,
        "temperature_duration": 1.0,
        "shock_anomaly": true,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [22]:
iot_test = [
    {
        "product_unit_id": 1,
        "timestamp": "2026-09-03T08:00:00Z",
        "temperature": 30.0,
        "accel_magnitude": 1.0
    },
    {
        "product_unit_id": 1,
        "timestamp": "2026-09-03T09:00:00Z",
        "temperature": 31.0,
        "accel_magnitude": 4.2
    }
]

iot_event = {
    "product_unit_id": 1,
    "event_type": "shipped",
    "actor_organization_id": 2,
    "_current_custodian_org_id": 2,
    "_unit_status": "in_transit",
    "latitude": 19.0760,
    "longitude": 72.8777,
    "timestamp": "2026-09-03T10:00:00Z",
    "_duplicate_scan_count": 1
}

result = score_event(
    iot_event,
    [],
    iot_test
)

print(json.dumps(result, indent=4))

{
    "risk_score": 60,
    "risk_level": "Medium",
    "reason": "Temperature excursion: 30.0\u00b0C outside configured range [18.0\u00b0C, 26.0\u00b0C]; Shock/handling anomaly: acceleration magnitude 4.20g exceeds configured threshold 3.0g",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 1,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": true,
        "temperature_duration": 1.0,
        "shock_anomaly": true,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [23]:
broken_event = None

result = score_event(
    broken_event,
    [],
    []
)

print(json.dumps(result, indent=4))

{
    "risk_score": 0,
    "risk_level": "Low",
    "reason": "Scoring unavailable; safe fallback returned",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 0,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1_fallback"
}


In [24]:
broken_event = None

result = score_event(
    broken_event,
    [],
    []
)

print(json.dumps(result, indent=4))

{
    "risk_score": 0,
    "risk_level": "Low",
    "reason": "Scoring unavailable; safe fallback returned",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 0,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1_fallback"
}


In [25]:
print("Product unit columns:")
print(product_units.columns.tolist())

Product unit columns:
['id', 'product_hash', 'qr_public_token', 'gtin', 'batch_no', 'serial_no', 'product_name', 'expiry_date', 'manufacturer_org_id', 'current_custodian_org_id', 'status', 'risk_score', 'risk_level', 'scan_count', 'last_verified_at', 'created_at']


In [26]:
unit_info = product_units[
    [
        "id",
        "current_custodian_org_id",
        "status"
    ]
].copy()

unit_info = unit_info.rename(
    columns={
        "id": "product_unit_id"
    }
)

lifecycle_ai = lifecycle_events.merge(
    unit_info,
    on="product_unit_id",
    how="left"
)

print("Merged lifecycle dataset:")
display(lifecycle_ai.head())

Merged lifecycle dataset:


,id,product_unit_id,actor_id,actor_organization_id,event_type,location_label,latitude,longitude,timestamp,event_hash,previous_event_hash,tx_hash,chain_status,idempotency_key,metadata,created_at,current_custodian_org_id,status
0,1,1,24,5,manufactured,Chicago Regional Warehouse,41.8781,-87.6298,2024-01-13 00:00:00+00:00,98585201e0f91f050f1483d3c1ac0f8a90967279e11bcd...,0000000000000000000000000000000000000000000000...,0x336f85821f33795c929776f120c21263986c40a481b4...,confirmed,unit-1-manufactured-20240113T000000-3286,"{""batch_no"": ""B9935"", ""line"": ""A""}",2024-01-13 00:05:00+00,22,delivered
1,2,1,12,4,sold,New York Fulfillment Center,40.7128,-74.0060,2024-08-01 05:00:00+00:00,a5d0992fc32b4192857c7dbbe3c95a9a33fd1487d843f1...,0000000000000000000000000000000000000000000000...,0x6bbcaae97c9c33eec47f5cb5a24987ba1a4925cc00fe...,confirmed,unit-1-sold-20240801T050000-2402,{},2024-08-01 05:02:00+00,22,delivered
2,3,1,14,3,manufactured,Mumbai Inspection Center,19.0760,72.8777,2025-01-07 11:00:00+00:00,2c33fdf7ddbdd4564fccbf203a4139e54ba582c1887723...,a5d0992fc32b4192857c7dbbe3c95a9a33fd1487d843f1...,NaN,pending,unit-1-manufactured-20250107T110000-6995,{},2025-01-07 11:04:00+00,22,delivered
3,4,1,17,5,manufactured,Mexico City Assembly Plant,19.4326,-99.1332,2025-05-15 09:00:00+00:00,a10aa79c84f6cb3f30d5cd7eab43064c1888daa8bea013...,2c33fdf7ddbdd4564fccbf203a4139e54ba582c1887723...,0x3e94a86f1fa8f987b9e02f1c8bd874a8cf122841e6fd...,failed,unit-1-manufactured-20250515T090000-8054,{},2025-05-15 09:00:00+00,22,delivered
4,5,2,2,1,manufactured,Los Angeles Port Terminal,33.7406,-118.2706,2024-08-04 00:00:00+00:00,32d102697fcac0f6be704b32ce5c246b1994a7c2458ef0...,0000000000000000000000000000000000000000000000...,0x08c48958d5f8eb4c1c284d2a5b425818d88c3a94b2a4...,confirmed,unit-2-manufactured-20240804T000000-4811,"{""batch_no"": ""B1434"", ""line"": ""C""}",2024-08-04 00:04:00+00,20,created


In [27]:
ML_FEATURES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

ml_data = ai_results.copy()

print("Available columns:")
print(ml_data.columns.tolist())

NameError: name 'ai_results' is not defined

In [29]:
# ============================================================
# CREATE AI RESULTS FROM ACTUAL CSV DATA
# ============================================================

results = []

# Make sure lifecycle events are sorted
lifecycle_ai = lifecycle_events.copy()

lifecycle_ai["timestamp"] = pd.to_datetime(
    lifecycle_ai["timestamp"],
    errors="coerce",
    utc=True
)

lifecycle_ai = lifecycle_ai.sort_values(
    ["product_unit_id", "timestamp"]
).reset_index(drop=True)

# Prepare product-unit information
unit_info_columns = ["id"]

if "current_custodian_org_id" in product_units.columns:
    unit_info_columns.append("current_custodian_org_id")

if "status" in product_units.columns:
    unit_info_columns.append("status")

unit_info = product_units[unit_info_columns].copy()

unit_info = unit_info.rename(
    columns={"id": "product_unit_id"}
)

# Merge product-unit information
lifecycle_ai = lifecycle_ai.merge(
    unit_info,
    on="product_unit_id",
    how="left"
)

# Make sure verification timestamps are datetime
if "created_at" in verification_events.columns:
    verification_events["created_at"] = pd.to_datetime(
        verification_events["created_at"],
        errors="coerce",
        utc=True
    )

# Process every lifecycle event
for idx, row in lifecycle_ai.iterrows():

    event_data = row.to_dict()

    unit_id = row["product_unit_id"]
    event_time = row["timestamp"]

    # --------------------------------------------------------
    # Current custodian
    # --------------------------------------------------------

    event_data["_current_custodian_org_id"] = row.get(
        "current_custodian_org_id"
    )

    # --------------------------------------------------------
    # Unit status
    # --------------------------------------------------------

    event_data["_unit_status"] = row.get(
        "status"
    )

    # --------------------------------------------------------
    # Previous lifecycle events
    # --------------------------------------------------------

    previous_rows = lifecycle_ai[
        (lifecycle_ai["product_unit_id"].astype(str) == str(unit_id))
        &
        (lifecycle_ai["timestamp"] < event_time)
    ].sort_values("timestamp")

    prior_events = (
        previous_rows
        .drop(
            columns=[
                "current_custodian_org_id",
                "status"
            ],
            errors="ignore"
        )
        .to_dict("records")
    )

    # --------------------------------------------------------
    # Verification scan count
    # --------------------------------------------------------

    verification_for_unit = verification_events[
        verification_events["product_unit_id"].astype(str)
        == str(unit_id)
    ].copy()

    if "created_at" in verification_for_unit.columns:
        verification_for_unit = verification_for_unit[
            verification_for_unit["created_at"] <= event_time
        ]

    event_data["_duplicate_scan_count"] = len(
        verification_for_unit
    )

    # --------------------------------------------------------
    # IoT readings
    # --------------------------------------------------------
    # No iot_readings.csv is currently available in the
    # six CSV files you showed, so use an empty list safely.
    
    iot_readings = []

    # --------------------------------------------------------
    # Run AI scoring
    # --------------------------------------------------------

    result = score_event(
        event_data,
        prior_events,
        iot_readings
    )

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    results.append({
        "product_unit_id": unit_id,
        "event_id": row.get("id"),
        "event_type": row.get("event_type"),
        "timestamp": row.get("timestamp"),
        "risk_score": result["risk_score"],
        "risk_level": result["risk_level"],
        "reason": result["reason"],
        **result["feature_values"]
    })


# Convert results into DataFrame
ai_results = pd.DataFrame(results)

print("======================================")
print("AI SCORING COMPLETED")
print("======================================")

print("Total events scored:", len(ai_results))

print("\nColumns:")
print(ai_results.columns.tolist())

print("\nFirst 10 results:")
display(ai_results.head(10))

AI SCORING COMPLETED
Total events scored: 300

Columns:
['product_unit_id', 'event_id', 'event_type', 'timestamp', 'risk_score', 'risk_level', 'reason', 'sequence_violation', 'geo_implausibility', 'duplicate_scan_count', 'time_gap_anomaly', 'unexpected_custodian', 'temperature_violation', 'temperature_duration', 'shock_anomaly', 'sensor_gap', 'sensor_provenance_mismatch']

First 10 results:


,product_unit_id,event_id,event_type,timestamp,risk_score,risk_level,reason,sequence_violation,geo_implausibility,duplicate_scan_count,time_gap_anomaly,unexpected_custodian,temperature_violation,temperature_duration,shock_anomaly,sensor_gap,sensor_provenance_mismatch
0,1,1,manufactured,2024-01-13 00:00:00+00:00,40,Medium,"Event logged by organization 5, but current cu...",False,False,0,False,True,False,0.0,False,False,False
1,1,2,sold,2024-08-01 05:00:00+00:00,65,High,"Event logged by organization 4, but current cu...",False,False,0,True,True,False,0.0,False,False,False
2,1,3,manufactured,2025-01-07 11:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...,True,False,1,True,True,False,0.0,False,False,False
3,1,4,manufactured,2025-05-15 09:00:00+00:00,100,High,QR code scanned 5 times — repeated scans may i...,True,False,5,True,True,False,0.0,False,False,False
4,2,5,manufactured,2024-08-04 00:00:00+00:00,40,Medium,"Event logged by organization 1, but current cu...",False,False,0,False,True,False,0.0,False,False,False
5,2,6,inspected,2024-08-05 08:00:00+00:00,40,Medium,"Event logged by organization 9, but current cu...",False,False,0,False,True,False,0.0,False,False,False
6,2,7,shipped,2024-08-07 06:00:00+00:00,40,Medium,"Event logged by organization 7, but current cu...",False,False,0,False,True,False,0.0,False,False,False
7,2,8,received,2024-08-10 03:00:00+00:00,40,Medium,"Event logged by organization 7, but current cu...",False,False,0,False,True,False,0.0,False,False,False
8,2,9,sold,2024-08-12 16:00:00+00:00,40,Medium,"Event logged by organization 9, but current cu...",False,False,0,False,True,False,0.0,False,False,False
9,2,10,shipped,2024-10-25 09:00:00+00:00,100,High,Event 'shipped' logged out of order — a later ...,True,False,0,True,True,False,0.0,False,False,False


In [30]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

NameError: name 'X' is not defined

In [31]:
print("ai_results columns:")
print(ai_results.columns.tolist())

ai_results columns:
['product_unit_id', 'event_id', 'event_type', 'timestamp', 'risk_score', 'risk_level', 'reason', 'sequence_violation', 'geo_implausibility', 'duplicate_scan_count', 'time_gap_anomaly', 'unexpected_custodian', 'temperature_violation', 'temperature_duration', 'shock_anomaly', 'sensor_gap', 'sensor_provenance_mismatch']


In [32]:
print("\nDoes label_anomalous exist?")
print("label_anomalous" in ai_results.columns)


Does label_anomalous exist?
False


In [33]:
print("========== ALL DATASET COLUMNS ==========")

datasets = {
    "organizations": organizations,
    "verification_events": verification_events,
    "lifecycle_events": lifecycle_events,
    "product_units": product_units,
    "provenance_flags": provenance_flags,
    "users": users
}

for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

========== ALL DATASET COLUMNS ==========

organizations:
['id', 'name', 'org_type', 'created_at']

verification_events:
['id', 'product_unit_id', 'result', 'ip_hash', 'user_agent_hash', 'created_at']

lifecycle_events:
['id', 'product_unit_id', 'actor_id', 'actor_organization_id', 'event_type', 'location_label', 'latitude', 'longitude', 'timestamp', 'event_hash', 'previous_event_hash', 'tx_hash', 'chain_status', 'idempotency_key', 'metadata', 'created_at']

product_units:
['id', 'product_hash', 'qr_public_token', 'gtin', 'batch_no', 'serial_no', 'product_name', 'expiry_date', 'manufacturer_org_id', 'current_custodian_org_id', 'status', 'risk_score', 'risk_level', 'scan_count', 'last_verified_at', 'created_at']

provenance_flags:
['id', 'product_unit_id', 'event_id', 'risk_score', 'reason', 'feature_values', 'model_version', 'created_at']

users:
['id', 'email', 'password_hash', 'role', 'organization_id', 'created_at']


In [35]:
ML_FEATURES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

X = ai_results[ML_FEATURES].copy()

y = ai_results["label_anomalous"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nLabel distribution:")
print(y.value_counts())


KeyError: 'label_anomalous'

In [36]:
ML_FEATURES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

X = ai_results[ML_FEATURES].copy()

y = ai_results["label_anomalous"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nLabel distribution:")
print(y.value_counts())

KeyError: 'label_anomalous'

In [37]:
print("Columns available in ai_results:")
print()

for column in ai_results.columns:
    print("-", column)

Columns available in ai_results:

- product_unit_id
- event_id
- event_type
- timestamp
- risk_score
- risk_level
- reason
- sequence_violation
- geo_implausibility
- duplicate_scan_count
- time_gap_anomaly
- unexpected_custodian
- temperature_violation
- temperature_duration
- shock_anomaly
- sensor_gap
- sensor_provenance_mismatch


In [38]:
print("\nDoes label_anomalous exist?")
print("label_anomalous" in ai_results.columns)


Does label_anomalous exist?
False


In [39]:
print("========== ALL DATASET COLUMNS ==========")

datasets = {
    "organizations": organizations,
    "verification_events": verification_events,
    "lifecycle_events": lifecycle_events,
    "product_units": product_units,
    "provenance_flags": provenance_flags,
    "users": users
}

for name, df in datasets.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

========== ALL DATASET COLUMNS ==========

organizations:
['id', 'name', 'org_type', 'created_at']

verification_events:
['id', 'product_unit_id', 'result', 'ip_hash', 'user_agent_hash', 'created_at']

lifecycle_events:
['id', 'product_unit_id', 'actor_id', 'actor_organization_id', 'event_type', 'location_label', 'latitude', 'longitude', 'timestamp', 'event_hash', 'previous_event_hash', 'tx_hash', 'chain_status', 'idempotency_key', 'metadata', 'created_at']

product_units:
['id', 'product_hash', 'qr_public_token', 'gtin', 'batch_no', 'serial_no', 'product_name', 'expiry_date', 'manufacturer_org_id', 'current_custodian_org_id', 'status', 'risk_score', 'risk_level', 'scan_count', 'last_verified_at', 'created_at']

provenance_flags:
['id', 'product_unit_id', 'event_id', 'risk_score', 'reason', 'feature_values', 'model_version', 'created_at']

users:
['id', 'email', 'password_hash', 'role', 'organization_id', 'created_at']


In [41]:
print("AI Results Shape:")
print(ai_results.shape)

print("\nAI Results Columns:")
print(ai_results.columns.tolist())

print("\nFirst 10 AI Results:")
display(ai_results.head(10))

AI Results Shape:
(300, 17)

AI Results Columns:
['product_unit_id', 'event_id', 'event_type', 'timestamp', 'risk_score', 'risk_level', 'reason', 'sequence_violation', 'geo_implausibility', 'duplicate_scan_count', 'time_gap_anomaly', 'unexpected_custodian', 'temperature_violation', 'temperature_duration', 'shock_anomaly', 'sensor_gap', 'sensor_provenance_mismatch']

First 10 AI Results:


,product_unit_id,event_id,event_type,timestamp,risk_score,risk_level,reason,sequence_violation,geo_implausibility,duplicate_scan_count,time_gap_anomaly,unexpected_custodian,temperature_violation,temperature_duration,shock_anomaly,sensor_gap,sensor_provenance_mismatch
0,1,1,manufactured,2024-01-13 00:00:00+00:00,40,Medium,"Event logged by organization 5, but current cu...",False,False,0,False,True,False,0.0,False,False,False
1,1,2,sold,2024-08-01 05:00:00+00:00,65,High,"Event logged by organization 4, but current cu...",False,False,0,True,True,False,0.0,False,False,False
2,1,3,manufactured,2025-01-07 11:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...,True,False,1,True,True,False,0.0,False,False,False
3,1,4,manufactured,2025-05-15 09:00:00+00:00,100,High,QR code scanned 5 times — repeated scans may i...,True,False,5,True,True,False,0.0,False,False,False
4,2,5,manufactured,2024-08-04 00:00:00+00:00,40,Medium,"Event logged by organization 1, but current cu...",False,False,0,False,True,False,0.0,False,False,False
5,2,6,inspected,2024-08-05 08:00:00+00:00,40,Medium,"Event logged by organization 9, but current cu...",False,False,0,False,True,False,0.0,False,False,False
6,2,7,shipped,2024-08-07 06:00:00+00:00,40,Medium,"Event logged by organization 7, but current cu...",False,False,0,False,True,False,0.0,False,False,False
7,2,8,received,2024-08-10 03:00:00+00:00,40,Medium,"Event logged by organization 7, but current cu...",False,False,0,False,True,False,0.0,False,False,False
8,2,9,sold,2024-08-12 16:00:00+00:00,40,Medium,"Event logged by organization 9, but current cu...",False,False,0,False,True,False,0.0,False,False,False
9,2,10,shipped,2024-10-25 09:00:00+00:00,100,High,Event 'shipped' logged out of order — a later ...,True,False,0,True,True,False,0.0,False,False,False


In [42]:
print("Shape of ai_results:", ai_results.shape)

Shape of ai_results: (300, 17)


In [43]:
print("========== RISK LEVEL DISTRIBUTION ==========")

print(
    ai_results["risk_level"].value_counts()
)

print("\n========== RISK SCORE STATISTICS ==========")

print(
    ai_results["risk_score"].describe()
)

========== RISK LEVEL DISTRIBUTION ==========
risk_level
High      175
Medium    119
Low         6
Name: count, dtype: int64

========== RISK SCORE STATISTICS ==========
count    300.000000
mean      68.653333
std       27.774651
min        0.000000
25%       40.000000
50%       79.500000
75%      100.000000
max      100.000000
Name: risk_score, dtype: float64


In [44]:
feature_columns = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

print("========== FEATURE ACTIVATION ==========")

for feature in feature_columns:

    if feature == "duplicate_scan_count":
        count = (ai_results[feature] > 1).sum()

    elif feature == "temperature_duration":
        count = (ai_results[feature] > 0).sum()

    else:
        count = (ai_results[feature] == True).sum()

    print(f"{feature}: {count}")

========== FEATURE ACTIVATION ==========
sequence_violation: 127
geo_implausibility: 14
duplicate_scan_count: 41
time_gap_anomaly: 101
unexpected_custodian: 288
temperature_violation: 0
temperature_duration: 0
shock_anomaly: 0
sensor_gap: 0
sensor_provenance_mismatch: 0


In [45]:
high_risk_events = ai_results.sort_values(
    "risk_score",
    ascending=False
)

display(
    high_risk_events[
        [
            "product_unit_id",
            "event_id",
            "event_type",
            "timestamp",
            "risk_score",
            "risk_level",
            "reason"
        ]
    ].head(20)
)

,product_unit_id,event_id,event_type,timestamp,risk_score,risk_level,reason
298,60,299,shipped,2024-09-25 22:00:00+00:00,100,High,Event 'shipped' logged out of order — a later ...
294,59,295,inspected,2024-03-17 07:00:00+00:00,100,High,Event 'inspected' logged out of order — a late...
291,59,292,manufactured,2024-02-09 00:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...
2,1,3,manufactured,2025-01-07 11:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...
264,53,265,manufactured,2025-05-13 00:00:00+00:00,100,High,QR code scanned 3 times — repeated scans may i...
35,6,36,received,2025-02-25 05:00:00+00:00,100,High,QR code scanned 2 times — repeated scans may i...
27,4,28,received,2024-12-30 16:00:00+00:00,100,High,Event 'received' logged out of order — a later...
23,4,24,manufactured,2024-07-26 21:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...
22,4,23,shipped,2024-07-07 13:00:00+00:00,100,High,Event 'shipped' logged out of order — a later ...
259,52,260,manufactured,2024-12-09 00:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...


In [46]:
print("========== 10 AI FEATURES ==========")

for feature in feature_columns:
    print(
        f"{feature:35} : "
        f"{feature in ai_results.columns}"
    )

========== 10 AI FEATURES ==========
sequence_violation                  : True
geo_implausibility                  : True
duplicate_scan_count                : True
time_gap_anomaly                    : True
unexpected_custodian                : True
temperature_violation               : True
temperature_duration                : True
shock_anomaly                       : True
sensor_gap                          : True
sensor_provenance_mismatch          : True


In [47]:
ai_results.to_csv(
    "ai_scored_lifecycle_events.csv",
    index=False
)

print("AI results saved successfully!")
print("File: ai_scored_lifecycle_events.csv")

AI results saved successfully!
File: ai_scored_lifecycle_events.csv


In [48]:
test_scores = [0, 34, 35, 64, 65, 100]

print("========== RISK LEVEL TEST ==========")

for score in test_scores:
    print(
        f"Score {score:3} -> {risk_level(score)}"
    )

========== RISK LEVEL TEST ==========
Score   0 -> Low
Score  34 -> Low
Score  35 -> Medium
Score  64 -> Medium
Score  65 -> High
Score 100 -> High


In [49]:
test_event = {
    "product_unit_id": 1,
    "event_type": "manufactured",
    "actor_organization_id": 1,
    "_current_custodian_org_id": 1,
    "_unit_status": "manufactured",
    "latitude": 19.0760,
    "longitude": 72.8777,
    "timestamp": "2026-09-01T10:00:00Z",
    "_duplicate_scan_count": 1
}

test_result = score_event(
    test_event,
    [],
    []
)

print(json.dumps(test_result, indent=4))

{
    "risk_score": 0,
    "risk_level": "Low",
    "reason": "No significant provenance anomaly detected",
    "feature_values": {
        "sequence_violation": false,
        "geo_implausibility": false,
        "duplicate_scan_count": 1,
        "time_gap_anomaly": false,
        "unexpected_custodian": false,
        "temperature_violation": false,
        "temperature_duration": 0.0,
        "shock_anomaly": false,
        "sensor_gap": false,
        "sensor_provenance_mismatch": false
    },
    "model_version": "deterministic_v1"
}


In [50]:
expected_fields = {
    "risk_score",
    "risk_level",
    "reason",
    "feature_values",
    "model_version"
}

actual_fields = set(test_result.keys())

print("Expected fields:", expected_fields)
print("Actual fields  :", actual_fields)

if actual_fields == expected_fields:
    print("\n✓ API structure is correct")
else:
    print("\n✗ API structure needs correction")

Expected fields: {'reason', 'feature_values', 'risk_level', 'risk_score', 'model_version'}
Actual fields  : {'reason', 'feature_values', 'risk_level', 'risk_score', 'model_version'}

✓ API structure is correct


In [51]:
expected_features = {
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
}

actual_features = set(
    test_result["feature_values"].keys()
)

print("Expected feature count:", len(expected_features))
print("Actual feature count  :", len(actual_features))

if actual_features == expected_features:
    print("\n✓ All 10 AI features are present")
else:
    print("\n✗ Feature structure needs correction")

Expected feature count: 10
Actual feature count  : 10

✓ All 10 AI features are present


In [57]:
ML_FEATURES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

ml_data = ai_results.copy()

print("Available columns:")
print(ml_data.columns.tolist())


Available columns:
['product_unit_id', 'event_id', 'event_type', 'timestamp', 'risk_score', 'risk_level', 'reason', 'sequence_violation', 'geo_implausibility', 'duplicate_scan_count', 'time_gap_anomaly', 'unexpected_custodian', 'temperature_violation', 'temperature_duration', 'shock_anomaly', 'sensor_gap', 'sensor_provenance_mismatch']


In [55]:
# ============================================================
# CHECK FOR SUPERVISED ML LABEL
# ============================================================

ML_FEATURES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

ml_data = ai_results.copy()

print("Available columns:")
print(ml_data.columns.tolist())

print("\nChecking for ground-truth label...")

if "label_anomalous" in ml_data.columns:
    print("✓ label_anomalous is available")
    print("Supervised ML can be trained.")
else:
    print("✗ label_anomalous is NOT available")
    print("Supervised ML cannot be trained from the current CSV data.")

Available columns:
['product_unit_id', 'event_id', 'event_type', 'timestamp', 'risk_score', 'risk_level', 'reason', 'sequence_violation', 'geo_implausibility', 'duplicate_scan_count', 'time_gap_anomaly', 'unexpected_custodian', 'temperature_violation', 'temperature_duration', 'shock_anomaly', 'sensor_gap', 'sensor_provenance_mismatch']

Checking for ground-truth label...
✗ label_anomalous is NOT available
Supervised ML cannot be trained from the current CSV data.


AI_FEATURES defined successfully!
Number of features: 10

Features:
1. sequence_violation
2. geo_implausibility
3. duplicate_scan_count
4. time_gap_anomaly
5. unexpected_custodian
6. temperature_violation
7. temperature_duration
8. shock_anomaly
9. sensor_gap
10. sensor_provenance_mismatch


In [67]:
print("========== 10 FEATURE CHECK ==========")

for feature in AI_FEATURES:
    print(
        f"{feature:35} : "
        f"{feature in ai_results.columns}"
    )

========== 10 FEATURE CHECK ==========
sequence_violation                  : True
geo_implausibility                  : True
duplicate_scan_count                : True
time_gap_anomaly                    : True
unexpected_custodian                : True
temperature_violation               : True
temperature_duration                : True
shock_anomaly                       : True
sensor_gap                          : True
sensor_provenance_mismatch          : True


In [68]:
print("========== RISK LEVEL DISTRIBUTION ==========")

print(
    ai_results[
        "risk_level"
    ].value_counts()
)

print(
    "\n========== RISK SCORE STATISTICS =========="
)

print(
    ai_results[
        "risk_score"
    ].describe()
)

========== RISK LEVEL DISTRIBUTION ==========
risk_level
High      175
Medium    119
Low         6
Name: count, dtype: int64

========== RISK SCORE STATISTICS ==========
count    300.000000
mean      68.653333
std       27.774651
min        0.000000
25%       40.000000
50%       79.500000
75%      100.000000
max      100.000000
Name: risk_score, dtype: float64


In [69]:
print("========== FEATURE ACTIVATION ==========")

for feature in AI_FEATURES:

    if feature == "duplicate_scan_count":

        count = (
            ai_results[feature] > 1
        ).sum()

    elif feature == "temperature_duration":

        count = (
            ai_results[feature] > 0
        ).sum()

    else:

        count = (
            ai_results[feature] == True
        ).sum()

    print(
        f"{feature:35} : {count}"
    )

========== FEATURE ACTIVATION ==========
sequence_violation                  : 127
geo_implausibility                  : 14
duplicate_scan_count                : 41
time_gap_anomaly                    : 101
unexpected_custodian                : 288
temperature_violation               : 0
temperature_duration                : 0
shock_anomaly                       : 0
sensor_gap                          : 0
sensor_provenance_mismatch          : 0


In [70]:
highest_risk = (
    ai_results
    .sort_values(
        "risk_score",
        ascending=False
    )
)

display(
    highest_risk[
        [
            "product_unit_id",
            "event_id",
            "event_type",
            "timestamp",
            "risk_score",
            "risk_level",
            "reason"
        ]
    ].head(20)
)

,product_unit_id,event_id,event_type,timestamp,risk_score,risk_level,reason
298,60,299,shipped,2024-09-25 22:00:00+00:00,100,High,Event 'shipped' logged out of order — a later ...
294,59,295,inspected,2024-03-17 07:00:00+00:00,100,High,Event 'inspected' logged out of order — a late...
291,59,292,manufactured,2024-02-09 00:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...
2,1,3,manufactured,2025-01-07 11:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...
264,53,265,manufactured,2025-05-13 00:00:00+00:00,100,High,QR code scanned 3 times — repeated scans may i...
35,6,36,received,2025-02-25 05:00:00+00:00,100,High,QR code scanned 2 times — repeated scans may i...
27,4,28,received,2024-12-30 16:00:00+00:00,100,High,Event 'received' logged out of order — a later...
23,4,24,manufactured,2024-07-26 21:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...
22,4,23,shipped,2024-07-07 13:00:00+00:00,100,High,Event 'shipped' logged out of order — a later ...
259,52,260,manufactured,2024-12-09 00:00:00+00:00,100,High,Event 'manufactured' logged out of order — a l...


In [71]:
ai_results.to_csv(
    "ai_scored_lifecycle_events.csv",
    index=False
)

print(
    "AI results saved successfully."
)

print(
    "File: ai_scored_lifecycle_events.csv"
)

AI results saved successfully.
File: ai_scored_lifecycle_events.csv


In [72]:
random.seed(42)
np.random.seed(42)

SYNTHETIC_ROWS = 5000
ANOMALY_RATE = 0.18

synthetic_rows = []

for i in range(SYNTHETIC_ROWS):

    is_anomalous = (
        random.random()
        < ANOMALY_RATE
    )

    row = {
        "sequence_violation": 0,
        "geo_implausibility": 0,
        "duplicate_scan_count": 0,
        "time_gap_anomaly": 0,
        "unexpected_custodian": 0,
        "temperature_violation": 0,
        "temperature_duration": 0.0,
        "shock_anomaly": 0,
        "sensor_gap": 0,
        "sensor_provenance_mismatch": 0
    }

    if is_anomalous:

        anomaly_type = random.choice([
            "sequence_violation",
            "geo_implausibility",
            "duplicate_scan_count",
            "time_gap_anomaly",
            "unexpected_custodian",
            "temperature_violation",
            "temperature_duration",
            "shock_anomaly",
            "sensor_gap",
            "sensor_provenance_mismatch"
        ])

        if anomaly_type == "sequence_violation":
            row["sequence_violation"] = 1

        elif anomaly_type == "geo_implausibility":
            row["geo_implausibility"] = 1

        elif anomaly_type == "duplicate_scan_count":
            row["duplicate_scan_count"] = random.randint(2, 5)

        elif anomaly_type == "time_gap_anomaly":
            row["time_gap_anomaly"] = 1

        elif anomaly_type == "unexpected_custodian":
            row["unexpected_custodian"] = 1

        elif anomaly_type == "temperature_violation":
            row["temperature_violation"] = 1

        elif anomaly_type == "temperature_duration":
            row["temperature_duration"] = round(
                random.uniform(
                    0.5,
                    5.0
                ),
                2
            )

        elif anomaly_type == "shock_anomaly":
            row["shock_anomaly"] = 1

        elif anomaly_type == "sensor_gap":
            row["sensor_gap"] = 1

        elif anomaly_type == "sensor_provenance_mismatch":
            row[
                "sensor_provenance_mismatch"
            ] = 1

    else:

        # Normal examples may still have
        # zero or one normal scan.
        row["duplicate_scan_count"] = random.choice(
            [0, 1]
        )

    row["label_anomalous"] = int(
        is_anomalous
    )

    synthetic_rows.append(row)


synthetic_df = pd.DataFrame(
    synthetic_rows
)

print(
    "Synthetic dataset shape:",
    synthetic_df.shape
)

display(
    synthetic_df.head()
)

Synthetic dataset shape: (5000, 11)


,sequence_violation,geo_implausibility,duplicate_scan_count,time_gap_anomaly,unexpected_custodian,temperature_violation,temperature_duration,shock_anomaly,sensor_gap,sensor_provenance_mismatch,label_anomalous
0,0,0,0,0,0,0,0.0,0,0,0,0
1,0,0,0,0,0,0,0.0,0,0,0,0
2,0,0,0,0,0,0,0.0,0,0,0,0
3,0,0,0,0,0,0,0.0,0,0,0,0
4,0,0,0,0,0,0,0.0,0,0,0,0


In [73]:
print(
    "========== SYNTHETIC LABEL DISTRIBUTION =========="
)

print(
    synthetic_df[
        "label_anomalous"
    ].value_counts()
)

print()

print(
    synthetic_df[
        "label_anomalous"
    ].value_counts(
        normalize=True
    )
)

========== SYNTHETIC LABEL DISTRIBUTION ==========
label_anomalous
0    4098
1     902
Name: count, dtype: int64

label_anomalous
0    0.8196
1    0.1804
Name: proportion, dtype: float64


In [74]:
ML_FEATURES = [
    "sequence_violation",
    "geo_implausibility",
    "duplicate_scan_count",
    "time_gap_anomaly",
    "unexpected_custodian",
    "temperature_violation",
    "temperature_duration",
    "shock_anomaly",
    "sensor_gap",
    "sensor_provenance_mismatch"
]

X = synthetic_df[
    ML_FEATURES
].copy()

y = synthetic_df[
    "label_anomalous"
].astype(int)

print(
    "X shape:",
    X.shape
)

print(
    "y shape:",
    y.shape
)

print(
    "\nLabel distribution:"
)

print(
    y.value_counts()
)

X shape: (5000, 10)
y shape: (5000,)

Label distribution:
label_anomalous
0    4098
1     902
Name: count, dtype: int64


In [75]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(
    "Training samples:",
    len(X_train)
)

print(
    "Testing samples:",
    len(X_test)
)

print(
    "\nTraining label distribution:"
)

print(
    y_train.value_counts()
)

print(
    "\nTesting label distribution:"
)

print(
    y_test.value_counts()
)

Training samples: 4000
Testing samples: 1000

Training label distribution:
label_anomalous
0    3278
1     722
Name: count, dtype: int64

Testing label distribution:
label_anomalous
0    820
1    180
Name: count, dtype: int64


In [83]:
from sklearn.linear_model import LogisticRegression

ml_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

ml_model.fit(
    X_train,
    y_train
)

print(
    "Logistic Regression trained successfully."
)

Logistic Regression trained successfully.


In [84]:
y_pred = ml_model.predict(
    X_test
)

print(
    "Predictions generated:"
)

print(
    y_pred[:20]
)

Predictions generated:
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]


In [85]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print(
    "========== ML EVALUATION =========="
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    "\nClassification Report:"
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print(
    "\nIMPORTANT:"
)

print(
    "These metrics are measured on synthetic data."
)

========== ML EVALUATION ==========
Precision: 1.0000
Recall   : 0.9278
F1 Score : 0.9625

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       820
           1       1.00      0.93      0.96       180

    accuracy                           0.99      1000
   macro avg       0.99      0.96      0.98      1000
weighted avg       0.99      0.99      0.99      1000


IMPORTANT:
These metrics are measured on synthetic data.


In [86]:
import joblib

model_dir = Path("ai_model")

model_dir.mkdir(
    exist_ok=True
)

model_path = (
    model_dir
    / "model.joblib"
)

joblib.dump(
    ml_model,
    model_path
)

print(
    "ML model saved successfully."
)

print(
    "Path:",
    model_path
)

ML model saved successfully.
Path: ai_model\model.joblib


In [87]:
loaded_model = joblib.load(
    model_path
)

loaded_predictions = loaded_model.predict(
    X_test
)

print(
    "Saved model loaded successfully."
)

print(
    "Predictions generated:",
    len(loaded_predictions)
)

Saved model loaded successfully.
Predictions generated: 1000


In [88]:
synthetic_df.to_csv(
    "synthetic_provenance_ml_dataset.csv",
    index=False
)

print(
    "Synthetic ML dataset saved."
)

print(
    "File: synthetic_provenance_ml_dataset.csv"
)

Synthetic ML dataset saved.
File: synthetic_provenance_ml_dataset.csv


In [89]:
print("=" * 60)
print("        SUPPLYCHAINX AI MODULE STATUS")
print("=" * 60)

print()

print("1. CSV DATA LOADED")
print("   ✓ organizations.csv")
print("   ✓ verification_events.csv")
print("   ✓ lifecycle_events.csv")
print("   ✓ product_units_seed.csv")
print("   ✓ provenance_flags CSV")
print("   ✓ users.csv")

print()

print("2. DETERMINISTIC ENGINE")
print("   ✓ 10 features implemented")
print("   ✓ Weighted scoring implemented")
print("   ✓ Risk levels implemented")
print("   ✓ Reason generation implemented")
print("   ✓ score_event() implemented")
print("   ✓ Fallback implemented")

print()

print("3. ACTUAL SEED DATA")
print("   ✓ Lifecycle events scored")
print("   ✓ Risk distribution calculated")
print("   ✓ Highest-risk events identified")
print("   ✓ Results exported")

print()

print("4. OPTIONAL ML")
print("   ✓ Synthetic dataset generated")
print("   ✓ label_anomalous available")
print("   ✓ 80/20 stratified split")
print("   ✓ Logistic Regression trained")
print("   ✓ Precision calculated")
print("   ✓ Recall calculated")
print("   ✓ F1 calculated")
print("   ✓ Model serialized")

print()

print("5. FINAL MODEL VERSION")
print("   deterministic_v1")
print("   logistic_synthetic_v1")

print()

print("=" * 60)
print("AI MODULE WORKFLOW COMPLETED")
print("=" * 60)

        SUPPLYCHAINX AI MODULE STATUS

1. CSV DATA LOADED
   ✓ organizations.csv
   ✓ verification_events.csv
   ✓ lifecycle_events.csv
   ✓ product_units_seed.csv
   ✓ provenance_flags CSV
   ✓ users.csv

2. DETERMINISTIC ENGINE
   ✓ 10 features implemented
   ✓ Weighted scoring implemented
   ✓ Risk levels implemented
   ✓ Reason generation implemented
   ✓ score_event() implemented
   ✓ Fallback implemented

3. ACTUAL SEED DATA
   ✓ Lifecycle events scored
   ✓ Risk distribution calculated
   ✓ Highest-risk events identified
   ✓ Results exported

4. OPTIONAL ML
   ✓ Synthetic dataset generated
   ✓ label_anomalous available
   ✓ 80/20 stratified split
   ✓ Logistic Regression trained
   ✓ Precision calculated
   ✓ Recall calculated
   ✓ F1 calculated
   ✓ Model serialized

5. FINAL MODEL VERSION
   deterministic_v1
   logistic_synthetic_v1

AI MODULE WORKFLOW COMPLETED
